# Echo Vest — Colab GPU Runner
**Before running this notebook:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Run `laptop_relay.py` on your laptop and copy the relay URL
3. Fill in your ngrok auth token and API keys in the config cell below

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 2: Clone repo and install dependencies ───────────────
import os

REPO_URL = 'https://github.com/Jrodd1203/echo-vest'  # update if needed
BRANCH   = 'front-end'

!git clone --branch {BRANCH} {REPO_URL} echo-vest 2>/dev/null || \
    (cd echo-vest && git fetch && git checkout {BRANCH} && git pull)

os.chdir('echo-vest')

!pip install -q -r python/requirements.txt
!pip install -q pyngrok flask

# OpenCV headless — no display needed on Colab
!pip install -q opencv-python-headless
!pip uninstall -q -y opencv-python 2>/dev/null || true

print('Dependencies installed.')

In [ ]:
# ── Cell 3: Config — fill these in ───────────────────────────

NGROK_AUTH_TOKEN = ''   # get free token at https://dashboard.ngrok.com/get-started/your-authtoken
CAMERA_RELAY_URL = ''   # paste the relay URL printed by laptop_relay.py  e.g. https://xxxx.ngrok-free.app

# API keys (must match your .env on the laptop)
OPENAI_API_KEY     = ''
ELEVENLABS_API_KEY = ''
DEEPGRAM_API_KEY   = ''

# Write a .env so python/main.py can load them
with open('python/.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write(f'ELEVENLABS_API_KEY={ELEVENLABS_API_KEY}\n')
    f.write(f'DEEPGRAM_API_KEY={DEEPGRAM_API_KEY}\n')

print('Config saved.')

In [ ]:
# ── Cell 4: Build React frontend ─────────────────────────────
!cd frontend && npm install --silent && npm run build
print('Frontend built.')

In [ ]:
# ── Cell 5: Start ngrok tunnels ───────────────────────────────
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# HTTP tunnel for the dashboard (port 8081)
dashboard_tunnel = ngrok.connect(8081, 'http')

# TCP tunnel for the motor WebSocket (port 8765)
# Update your ESP32 firmware to connect to this address instead of the laptop IP
motor_tunnel = ngrok.connect(8765, 'tcp')

print('\n' + '='*60)
print('  DASHBOARD:', dashboard_tunnel.public_url)
print('  MOTOR WS: ', motor_tunnel.public_url.replace('tcp://', 'ws://'))
print()
print('  Open the dashboard URL in your browser.')
print('  Update ESP32 firmware WebSocket host to the motor WS address.')
print('='*60 + '\n')

In [ ]:
# ── Cell 6: Run Echo Vest ─────────────────────────────────────
# SKIP_VOICE=1 because Colab has no microphone.
# Voice runs separately on the laptop via: python python/voice_standalone.py
import os, subprocess, sys, signal

# Kill any orphaned processes still holding our ports
subprocess.run(["fuser", "-k", "8080/tcp"], capture_output=True)
subprocess.run(["fuser", "-k", "8765/tcp"], capture_output=True)

env = os.environ.copy()
env['CAMERA_URL'] = CAMERA_RELAY_URL + '/stream'
env['SKIP_VOICE'] = '1'
env['SKIP_DISPLAY'] = '1'

os.chdir('/content/echo-vest/python')

proc = subprocess.Popen(
    [sys.executable, 'main.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream output line by line — interrupt the cell to stop
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('Server stopped.')
